# Day 3 — Constraint exploration

Four trade studies against the constrained landing problem. The question each
one answers is the question a mission designer actually asks: *which constraint
is binding, and what does it cost?*

| | |
|---|---|
| **A** | What does tightening the glideslope cost? |
| **B** | What does the pointing limit cost? |
| **C** | How does fuel scale with initial downrange? |
| **D** | Is there a best burn duration? |

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.landing_problem import (solve_landing, plot_landing,
                                 feasible_entry_state, max_downrange,
                                 min_arrestable_speed)
from src.dynamics import Vehicle

veh = Vehicle()

# Hold the entry state fixed across A, B and C so only one thing varies at a time.
Z0, VZ0 = feasible_entry_state(veh, t_burn=20.0, theta_max_deg=30.0)
print(f'entry state: z0 = {Z0:,.0f} m, vz0 = {VZ0:.1f} m/s')

v_req, drop_min, _ = min_arrestable_speed(veh, 20.0, 30.0)
print(f'minimum arrestable entry speed for a 20 s burn: {v_req:.1f} m/s')

def run(**kw):
    kw.setdefault('z0', Z0); kw.setdefault('vz0', VZ0)
    return solve_landing(verbose=False, **kw)

## Experiment A — what does the glideslope cost?

Sweep the cone from a permissive 50° to a near-vertical 88°, holding the entry
point at 150 m downrange.

In [ ]:
gammas = [50., 60., 70., 80., 84., 86., 87., 88.]
rows = []
for g in gammas:
    r = run(x0=150.0, gamma_gs_deg=g)
    ok = r['status'].startswith('optimal')
    rows.append((g, max_downrange(Z0, g), r['fuel'] if ok else np.nan, r['status']))
    print(f"gamma={g:>5.1f}  corridor at entry = {rows[-1][1]:>8,.0f} m   "
          f"fuel = {rows[-1][2]:>10,.0f} kg   [{r['status']}]")

**The result is flat, then a cliff.** Fuel does not move at all between 50° and
86°, then the problem goes infeasible at 88°.

That is worth sitting with. The glideslope is not a *fuel* constraint here, it is
a *feasibility* constraint. It costs nothing right up until the corridor at entry
narrows below the vehicle's starting offset (at 88° the corridor is 102 m and the
vehicle starts at 150 m), at which point there is no trajectory at all.

Constraints that behave this way are the dangerous ones in mission design: no
warning in the cost function, then a hard wall.

## Experiment B — what does the pointing limit cost?

In [ ]:
for th in [5., 8., 10., 15., 20., 30., 45., 60.]:
    r = run(x0=150.0, gamma_gs_deg=80.0, theta_max_deg=th)
    fuel = f"{r['fuel']:>10,.0f} kg" if r['status'].startswith('optimal') else f"{r['status']:>13}"
    print(f'theta_max={th:>5.1f} deg   {fuel}')

Same shape: essentially free above 20°, a few kg at 10°, infeasible at 5°.

The reason both look like this is that the vehicle barely steers. Its horizontal
excursion is small relative to its altitude, so the thrust vector spends the
whole burn within a few degrees of vertical — go back and look at the pointing
panel of `day3_landing.png`. A limit only costs something once it is smaller than
the angle the trajectory wanted to use anyway.

## Experiment C — fuel vs initial downrange

The plan predicts fuel rises roughly linearly with `x0`, since the vehicle needs
proportionally more horizontal impulse. Worth checking.

In [ ]:
print(f'glideslope corridor at z0 = {Z0:,.0f} m is |x| <= {max_downrange(Z0, 80.0):,.0f} m\n')
xs, fuels = [], []
for x0 in np.arange(0.0, 601.0, 50.0):
    r = run(x0=float(x0), gamma_gs_deg=80.0)
    ok = r['status'].startswith('optimal')
    print(f"x0={x0:>6.0f} m   " +
          (f"fuel = {r['fuel']:>10,.0f} kg" if ok else f"{r['status']}"))
    if ok:
        xs.append(x0); fuels.append(r['fuel'])

plt.figure(figsize=(7,4))
plt.plot(xs, fuels, 'o-', lw=2)
plt.xlabel('Initial downrange $x_0$ [m]'); plt.ylabel('Fuel [kg]')
plt.title('Fuel vs initial downrange'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**It is not linear, and it is not even monotone.** Fuel is a shallow U with a
minimum around `x0 ≈ 400 m`, roughly 170 kg *below* the cost of starting directly
over the pad.

The reason is that `vx0` is held at −40 m/s. Starting at `x0 = 0` the vehicle is
already drifting away from the pad and has to spend impulse turning that around.
Starting a few hundred metres downrange, the drift it already has carries it to
the pad for free. The cheapest entry is the one where the existing velocity
points where you want to go — which is why real entry targeting solves for
position and velocity together rather than treating downrange as an error to
minimise.

## Experiment D — is there a best burn duration?

Here the entry state has to be resized for each burn time: a longer burn can only
null a faster entry, and it has to start higher up.

In [ ]:
tbs, fs = [], []
for tb in [8., 10., 12., 15., 20., 25., 30., 32., 34., 36., 40.]:
    r = solve_landing(t_burn=tb, verbose=False)
    z, v = feasible_entry_state(veh, tb, 30.0)
    ok = r['status'].startswith('optimal')
    print(f"t_burn={tb:>5.1f} s  entry ({z:>7,.0f} m, {v:>7.1f} m/s)  " +
          (f"fuel = {r['fuel']:>10,.0f} kg" if ok else f"{r['status']}"))
    if ok:
        tbs.append(tb); fs.append(r['fuel'])

mdot_min = veh.T_min / (veh.isp * 9.80665)
print(f'\nminimum-throttle flow          = {mdot_min:,.0f} kg/s')
print(f'max burn on {veh.m_prop_initial:,.0f} kg of propellant = '
      f'{veh.m_prop_initial/mdot_min:.1f} s')

plt.figure(figsize=(7,4))
plt.plot(tbs, fs, 'o-', lw=2)
plt.axvline(veh.m_prop_initial/mdot_min, color='r', ls=':',
            label='propellant-limited burn duration')
plt.xlabel('Burn time [s]'); plt.ylabel('Fuel [kg]')
plt.title('Fuel vs burn duration'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**There is no sweet spot — fuel rises monotonically, and then the problem dies.**

The plan suggests a longer burn eventually wastes fuel on gravity losses while a
shorter one is infeasible, implying an interior optimum. That is not what this
vehicle does. Fuel is very nearly linear in burn time, and the hard limit is
arithmetic rather than a trade: minimum throttle flows 861 kg/s, so 30,000 kg of
propellant buys **34.9 seconds of burn, full stop**. Ask for 34 s or more and the
vehicle runs dry before touchdown.

The shortest feasible burn is always the cheapest here, and the real constraint
on how late you can ignite is structural and navigational, not propulsive. That
is the same conclusion Day 2 reached from the other direction: this vehicle
cannot throttle low enough to loiter, so every landing is a single committed
burn as late as you dare.

## Deliberate infeasibility

The diagnostic that matters is not "why won't it solve" but "which constraint
made it impossible". Relax one thing at a time.

In [ ]:
impossible = dict(x0=5000.0, z0=500.0, vx0=-200.0, vz0=-50.0, t_burn=15.0)
print('baseline           ->', solve_landing(**impossible, verbose=False)['status'])

trials = {
    'A: more time (40 s)':      dict(impossible, t_burn=40.0),
    'B: more altitude (3 km)':  dict(impossible, z0=3000.0, t_burn=20.0),
    'C: flatter glideslope':    dict(impossible, t_burn=20.0, gamma_gs_deg=50.0),
    'D: looser pointing':       dict(impossible, t_burn=20.0, theta_max_deg=60.0),
}
for label, kw in trials.items():
    r = solve_landing(verbose=False, **kw)
    ok = r['status'].startswith('optimal')
    print(f"{label:<26} -> {r['status']:<12}" +
          (f"  fuel = {r['fuel']:,.0f} kg" if ok else ''))

print('\nGeometry check: at z0 = 500 m the 80 deg corridor allows '
      f'|x| <= {max_downrange(500.0, 80.0):,.0f} m, but x0 = 5,000 m.')

The entry point is outside the glideslope cone *before the dynamics are even
considered* — 5,000 m downrange at 500 m altitude needs a corridor of 88 m. No
amount of thrust or time fixes that; only changing the geometry does. Run the
cheap geometric check first and you save yourself the solver.